# Treino RF e GB por área com os hiperparâmetros ajustados

Monta a estrutura para treinar RandomForest e GradientBoosting por componente em cada uma das sete
áreas com os hiperparâmetros de `notebooks/colabs_S4/artefatos/hiperparametros.json`, para depois
comparar com o baseline de `baseline_metricas_por_area.ipynb`.

A leitura dos dados, a partição, as 17 dobras leave-one-group-out, o tratamento de ausentes e as
funções de métrica seguem `treino_rf_gb_por_area.ipynb` e `baseline_metricas_por_area.ipynb`, de modo
que a única diferença entre o baseline e esta execução seja a parametrização dos modelos.

Nesta etapa nenhum modelo é treinado: as seções abaixo apenas preparam os dados, as métricas e a
tabela de hiperparâmetros.

## 1. Imports e caminhos

Mesma função `resolve` dos notebooks anteriores, que procura os arquivos de dados nas pastas usuais
do repositório. Os dados não são versionados, por isso o caminho não pode ser fixo.

In [ ]:
# == caminhos do projeto ==
# A pasta deste caderno, dentro de notebooks/. Os CSVs sao lidos e gravados
# nela; as leituras entre pastas usam ../<outra pasta>. Para que esses caminhos
# valham qualquer que seja o diretorio em que o kernel foi aberto, o bloco
# abaixo localiza a raiz do repositorio -- a pasta que contem notebooks/ e
# documents/ -- e entra na pasta do caderno. No Colab, onde nao existe
# repositorio e os arquivos ficam soltos ao lado do caderno, nada e alterado.
import os

PASTA_CADERNO = "colabs_S4/testes_extras/testes_treinamento"


def raiz_repositorio(inicio=None):
    """Sobe a arvore de diretorios ate a raiz do repositorio; None se nao achar."""
    atual = os.path.abspath(inicio or os.getcwd())
    while True:
        if all(os.path.isdir(os.path.join(atual, p)) for p in ("notebooks", "documents")):
            return atual
        pai = os.path.dirname(atual)
        if pai == atual:
            return None
        atual = pai


RAIZ_REPO = raiz_repositorio()
if RAIZ_REPO:
    os.chdir(os.path.join(RAIZ_REPO, "notebooks", PASTA_CADERNO))
print("Diretorio de trabalho:", os.path.abspath("."))

In [ ]:
import os, json, sys, warnings

import pandas as pd
import numpy as np

warnings.filterwarnings("ignore", category=UserWarning, module=r"sklearn|joblib")   # avisos de paralelismo

SUFIXOS = ("_perfil", "_13cat")
PASTA_DADOS     = "colabs_S3"
PASTA_PARTICAO  = "colabs_S3/preparacao_testes"

BASES = [
    ".", "..", os.path.join("..", ".."), os.path.join("..", "..", ".."), "/content",
    "notebooks", os.path.join("..", "notebooks"),
    os.path.join("notebooks", PASTA_DADOS),
    os.path.join("notebooks", PASTA_PARTICAO),
]

def resolve(pasta, arquivo):
    tentativas = []
    # A pasta declarada tem prioridade sobre o arquivo solto: sem isso, um
    # homonimo numa pasta vizinha pode ser lido no lugar do arquivo pedido.
    for partes in ((pasta, arquivo), (arquivo,)):
        for base in BASES:
            caminho = os.path.join(base, *partes)
            tentativas.append(caminho)
            if os.path.exists(caminho):
                return caminho
    raise FileNotFoundError(
        "Arquivo nao encontrado: " + arquivo
        + ". Diretorio atual: " + os.path.abspath(".")
        + ". Caminhos tentados: " + "; ".join(tentativas)
    )

print("Diretorio:", os.path.abspath("."))

## 2. Leitura da partição e das dobras

A partição é lida de arquivo, nunca sorteada. Os ensaios do conjunto de teste recebem dobra ausente
e ficam fora de `DOBRAS`; é essa condição que `indices_da_dobra` usa para descartá-los. Os asserts
tornam a garantia explícita, em vez de deixá-la implícita na estrutura do arquivo.

In [ ]:
with open(resolve(PASTA_PARTICAO, "particao_treino_teste.json"), encoding="utf-8") as f:
    particao = json.load(f)

IDX_TREINO = particao["run_idx_treino"]
IDX_TESTE  = particao["run_idx_teste"]
DOBRA  = {int(k): v for k, v in particao["dobra_por_run_idx"].items()}
DOBRAS = sorted({d for d in DOBRA.values() if d is not None})

RUN_TESTE = set(IDX_TESTE)

# O conjunto de teste reservado so esta protegido enquanto todo ensaio de teste tiver
# dobra ausente: e essa condicao que indices_da_dobra usa para descarta-lo.
assert all(DOBRA[r] is None for r in RUN_TESTE), "ensaio de teste com dobra atribuida"
assert {r for r, d in DOBRA.items() if d is not None} == set(IDX_TREINO), \
    "os ensaios com dobra nao coincidem com o conjunto de treino"
assert len(DOBRAS) == particao["n_dobras"] == 17, "a particao deveria ter 17 dobras de treino"

print(f"{len(IDX_TREINO)} ensaios de treino em {len(DOBRAS)} dobras | "
      f"{len(IDX_TESTE)} ensaios de teste reservados e nao lidos")
print("dobras:", DOBRAS)

As duas versões de perdas são carregadas juntas, como no notebook de referência: `_perfil` e
`_13cat` diferem apenas na codificação do motivo do ensaio, e o baseline reporta as duas.

In [ ]:
AREAS = ["coil_temps", "ct_knobs", "ipc_comp", "ipc_cooling",
         "main_vent_sys", "mainback", "oven_hmi"]

PREFIXOS_FEATURES = [
    "blower_direction", "housing", "board_position", "run_case",
    "losses", "perfil_perdas", "deflector_hole", "material_coil_support",
    "cfg",   # sem este prefixo, os atributos fisicos seriam tratados como alvo
]
COLUNAS_AUXILIARES = ["run_idx"]

def separar(df, prefixos=PREFIXOS_FEATURES):
    """Devolve (X, cols_alvo); run_idx fica fora de ambos."""
    cols_entrada = [c for c in df.columns if any(c.startswith(p + "_") for p in prefixos)]
    cols_alvo    = [c for c in df.columns if c not in cols_entrada + COLUNAS_AUXILIARES]
    return df[cols_entrada].copy(), cols_alvo


def carregar(sufixo):
    ds = {area: pd.read_csv(resolve(PASTA_DADOS, f"dataset_{area}{sufixo}.csv")) for area in AREAS}
    sep = {area: dict(zip(("X", "cols_alvo"), separar(df))) for area, df in ds.items()}
    return ds, sep

DADOS = {sufixo: carregar(sufixo) for sufixo in SUFIXOS}
limites = pd.read_csv(resolve(PASTA_DADOS, "limites_ativos.csv")).set_index("tc_name")

pd.DataFrame({
    (sufixo, area): {"linhas": DADOS[sufixo][0][area].shape[0],
                     "features": DADOS[sufixo][1][area]["X"].shape[1],
                     "alvos": len(DADOS[sufixo][1][area]["cols_alvo"])}
    for sufixo in SUFIXOS for area in AREAS
}).T

O tratamento de ausentes é por máscara: uma linha só entra no ajuste ou na validação de um
componente se aquele componente foi medido naquele ensaio. A seleção fica concentrada em
`indices_da_dobra`, idêntica à de `treino_rf_gb_por_area.ipynb`.

Os datasets tratados não trazem a coluna `run_idx`: o identificador do ensaio é a ordem das linhas,
que é o que permite associá-las à partição. Daí o fallback `np.arange(len(df))` e o assert de
alinhamento, sem o qual a associação entre linha e dobra seria silenciosamente errada.

A conferência percorre as dobras sem ajustar modelo algum, só para confirmar que a máscara cobre
exatamente os pares medidos de treino e que nenhum ensaio de teste é alcançado.

In [ ]:
def indices_da_dobra(df, dobra, y_col):
    """(idx_treino, idx_validacao) posicionais; o teste reservado nunca entra."""
    run = df["run_idx"].to_numpy() if "run_idx" in df.columns else np.arange(len(df))
    medido    = y_col.notna().to_numpy()
    no_treino = np.array([r in DOBRA and DOBRA[r] is not None for r in run])
    idx_val = [i for i in range(len(df)) if medido[i] and no_treino[i] and DOBRA[run[i]] == dobra]
    idx_tr  = [i for i in range(len(df)) if medido[i] and no_treino[i] and DOBRA[run[i]] != dobra]
    return idx_tr, idx_val


# Os datasets tratados nao trazem a coluna run_idx: o run_idx e a ordem das linhas, que e
# o que permite associar cada linha a particao. Por isso o alinhamento e conferido antes
# de qualquer uso, como faz random_search.ipynb.
for sufixo in SUFIXOS:
    for area in AREAS:
        assert len(DADOS[sufixo][0][area]) == len(DOBRA), \
            f"{area}{sufixo}: dataset e particao divergem em numero de ensaios"
print("datasets alinhados com a particao:", len(DOBRA), "ensaios em cada")


def run_idx_de(df):
    """Mesmo criterio de indices_da_dobra: coluna quando existe, ordem das linhas quando nao."""
    return df["run_idx"].to_numpy() if "run_idx" in df.columns else np.arange(len(df))


# Conferencia da mascara e da exclusao do teste, sem ajustar modelo algum: o total de linhas
# de validacao empilhadas sobre as 17 dobras deve igualar o numero de pares
# (ensaio de treino, componente) efetivamente medidos, e nenhum ensaio de teste pode aparecer.
conferencia, vistos = [], set()
for sufixo in SUFIXOS:
    ds, sep = DADOS[sufixo]
    for area in AREAS:
        df = ds[area]
        run = run_idx_de(df)
        n_val = n_medido = 0
        for col in sep[area]["cols_alvo"]:
            y = df[col]
            n_medido += int((y.notna().to_numpy()
                             & np.array([DOBRA.get(r) is not None for r in run])).sum())
            for d in DOBRAS:
                _, idx_val = indices_da_dobra(df, d, y)
                n_val += len(idx_val)
                vistos.update(run[i] for i in idx_val)
        conferencia.append({"sufixo": sufixo, "area": area,
                            "linhas_validacao": n_val, "pares_medidos_treino": n_medido,
                            "confere": n_val == n_medido})

assert not (vistos & RUN_TESTE), f"ensaio de teste alcancado: {sorted(vistos & RUN_TESTE)}"

conferencia = pd.DataFrame(conferencia)
assert conferencia["confere"].all(), "a mascara nao cobre exatamente os pares medidos de treino"
print("nenhum ensaio de teste alcancado |",
      f"{len(vistos)} ensaios distintos na validacao, todos de treino")
print("linhas de validacao fora-da-dobra por sufixo:",
      conferencia.groupby("sufixo")["linhas_validacao"].sum().to_dict())
conferencia

## 3. Métricas

Célula canônica de `colabs_S3/preparacao_testes/metricas.ipynb` (seção 1, "Definição das funções"), copiada sem
alteração. É a definição única de métrica do projeto: qualquer correção é feita lá e propagada às
cópias. Usar a mesma célula é o que torna estes números comparáveis aos do baseline.

In [ ]:
import os
import json

import numpy as np
import pandas as pd


MARGEM_KPI = 0.10   # margem de erro acordada no TAPI


def _valida(y_real, y_previsto):
    """Converte para vetores e recusa entradas que produziriam metrica sem sentido.

    A conversao para numpy e necessaria porque a subtracao entre duas Series do pandas
    alinha pelos indices, e nao pela posicao. Uma Series com indice original e outra
    com indice reiniciado produziriam diferencas entre pares errados, em silencio.

    O alvo precisa ser estritamente positivo. A restricao e legitima neste projeto,
    pois trata-se de temperatura absoluta em graus Celsius, cujo minimo na base e de
    27,69 graus. Ela existe para transformar em erro visivel o engano de passar valores
    padronizados, caso em que o alvo se aproximaria de zero e o erro percentual
    explodiria em vez de acusar o problema.
    """
    y_real = np.asarray(y_real, dtype=float).ravel()
    y_previsto = np.asarray(y_previsto, dtype=float).ravel()

    if y_real.shape != y_previsto.shape:
        raise ValueError(f"dimensoes diferentes: {y_real.shape} e {y_previsto.shape}")
    if y_real.size == 0:
        raise ValueError("nenhuma observacao a avaliar")
    if np.isnan(y_real).any():
        raise ValueError("ha valores ausentes em y_real; a mascara deve ser aplicada antes")
    if np.isnan(y_previsto).any():
        raise ValueError("ha valores ausentes em y_previsto")
    if (y_real <= 0).any():
        raise ValueError(
            "y_real contem valor nao positivo; o erro percentual exige temperatura "
            "absoluta em graus Celsius, e nao valores padronizados")

    return y_real, y_previsto


def avaliar(y_real, y_previsto, limite=None):
    """Metricas de erro de um componente, com o alvo em graus Celsius.

    Sempre devolvidas:

    n            numero de observacoes que sustentam as metricas
    MAE          erro absoluto medio, na unidade do alvo
    RMSE         raiz do erro quadratico medio, na mesma unidade
    R2           fracao da variancia do alvo explicada pelo modelo
    vies         erro medio com sinal, positivo quando o modelo superestima
    RMSE/MAE     razao entre as duas, indicando concentracao dos erros
    MAPE_%       erro percentual absoluto medio
    WAPE_%       erro absoluto total dividido pela soma dos alvos
    dentro_10_%  proporcao de predicoes dentro da margem de 10 por cento do TAPI

    Devolvidas apenas quando um limite de seguranca e informado:

    limite_C          o proprio limite, para conferencia
    n_reprova_real    ensaios em que o componente de fato atinge o limite
    acerto_decisao_%  concordancia entre a decisao real e a prevista
    falso_aprovado    ensaios reprovados na realidade e aprovados pelo modelo
    falso_reprovado   ensaios aprovados na realidade e reprovados pelo modelo

    O R2 sai ausente quando o alvo assume um unico valor, situacao em que ele e
    indefinido e em que a implementacao do scikit-learn devolve zero, valor que se
    confunde com um resultado legitimo.
    """
    y_real, y_previsto = _valida(y_real, y_previsto)

    erro = y_previsto - y_real
    mae = float(np.mean(np.abs(erro)))

    # A raiz e calculada explicitamente: o parametro squared do mean_squared_error foi
    # removido do scikit-learn na versao 1.9, e o root_mean_squared_error so existe a
    # partir da 1.4. O calculo direto funciona em qualquer versao.
    rmse = float(np.sqrt(np.mean(erro ** 2)))

    if np.unique(y_real).size < 2:
        r2 = float("nan")
    else:
        ss_res = float(np.sum(erro ** 2))
        ss_tot = float(np.sum((y_real - np.mean(y_real)) ** 2))
        r2 = 1.0 - ss_res / ss_tot

    relativo = np.abs(erro) / y_real

    resultado = {
        "n": int(y_real.size),
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
        "vies": float(np.mean(erro)),
        "RMSE/MAE": (rmse / mae) if mae > 0 else float("nan"),
        "MAPE_%": 100.0 * float(np.mean(relativo)),
        "WAPE_%": 100.0 * float(np.sum(np.abs(erro)) / np.sum(y_real)),
        "dentro_10_%": 100.0 * float(np.mean(relativo <= MARGEM_KPI)),
    }

    if limite is not None and np.isfinite(limite):
        reprova_real = y_real >= limite
        reprova_prevista = y_previsto >= limite
        resultado.update({
            "limite_C": float(limite),
            "n_reprova_real": int(reprova_real.sum()),
            "acerto_decisao_%": 100.0 * float(np.mean(reprova_real == reprova_prevista)),
            "falso_aprovado": int(np.sum(reprova_real & ~reprova_prevista)),
            "falso_reprovado": int(np.sum(~reprova_real & reprova_prevista)),
        })

    return resultado


def consolidar(tabela, grupo="dataset",
               metricas=("MAE", "RMSE", "R2", "vies", "MAPE_%", "WAPE_%", "dentro_10_%")):
    """Consolida metricas por componente em metricas por area.

    A macro-media atribui o mesmo peso a cada componente e responde se o modelo funciona
    bem em todas as partes do produto. A micro-media pondera pelo numero de observacoes
    e responde qual o erro tipico de uma leitura qualquer.

    O R2 nao e comparavel entre areas, pois seu denominador e a variancia do alvo de
    cada area. Ele indica se o modelo supera a media dentro da propria area.
    """
    if grupo not in tabela.columns:
        raise ValueError(f"coluna de agrupamento ausente: {grupo}")
    if "n" not in tabela.columns:
        raise ValueError("a tabela precisa da coluna n para a micro-media")

    metricas = [m for m in metricas if m in tabela.columns]
    linhas = []

    for nome, bloco in tabela.groupby(grupo):
        linha = {grupo: nome, "componentes": len(bloco), "n_total": int(bloco["n"].sum())}
        for m in metricas:
            valores = bloco[m]
            validos = valores.notna()
            linha[f"{m}_macro"] = (float(valores[validos].mean())
                                   if validos.any() else float("nan"))
            if validos.any() and bloco.loc[validos, "n"].sum() > 0:
                linha[f"{m}_micro"] = float(
                    np.average(valores[validos], weights=bloco.loc[validos, "n"]))
            else:
                linha[f"{m}_micro"] = float("nan")

        # As metricas de decisao sao contagens, e por isso somadas em vez de promediadas
        for contagem in ("n_reprova_real", "falso_aprovado", "falso_reprovado"):
            if contagem in bloco.columns:
                linha[contagem] = int(bloco[contagem].fillna(0).sum())
        linhas.append(linha)

    return pd.DataFrame(linhas).sort_values(grupo).reset_index(drop=True)


print("Metricas definidas: avaliar, consolidar")

## 4. Artefato de hiperparâmetros

`hiperparametros.py` lê o JSON gravado pela Parte III de `random_search.ipynb`. `params` devolve o
dicionário do par área/algoritmo e `modelo` já instancia o estimador correspondente.

O caminho é `../../artefatos` porque este notebook está em `notebooks/colabs_S4/testes_extras/testes_treinamento/`;
o exemplo do docstring do módulo, `../artefatos`, vale para um notebook dentro de `colabs_S4/`.

In [ ]:
sys.path.append("../../artefatos")
from hiperparametros import params, modelo

import hiperparametros
arte = hiperparametros.carrega()

print("gerado em:", arte["gerado_em"], "| origem:", arte["origem"])
print("formulacao da busca:", arte["protocolo"]["formulacao"])
print("validacao:", arte["protocolo"]["validacao"])
print("conjunto de teste:", arte["protocolo"]["conjunto_teste"])
print()
print("RF coil_temps:", params("coil_temps", "random_forest"))
print("GB coil_temps:", params("coil_temps", "gradient_boosting"))
print()
print("modelo() RF:", modelo("coil_temps", "random_forest"))
print("modelo() GB:", modelo("coil_temps", "gradient_boosting"))

## 5. Hiperparâmetros por área e diferença para o baseline

`treino_rf_gb_por_area.ipynb` declara apenas `{"rf": {"n_estimators": 300, "random_state": 42,
"n_jobs": -1}, "gb": {"random_state": 42}}` e deixa o restante no padrão do scikit-learn. A comparação
parâmetro a parâmetro só faz sentido contra o valor efetivo, e por isso o valor do baseline é lido de
`get_params()` do estimador construído com aquele dicionário, e não do dicionário em si. A coluna
`origem_baseline` registra se o valor foi declarado no notebook ou herdado do padrão.

Nenhum modelo é ajustado aqui: os estimadores são apenas instanciados para resolver os padrões.

In [ ]:
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

# Os mesmos de treino_rf_gb_por_area.ipynb, repetidos no registro de baseline_metricas_por_area.ipynb.
HIPERPARAMETROS_BASELINE = {
    "rf": {"n_estimators": 300, "random_state": 42, "n_jobs": -1},
    "gb": {"random_state": 42},
}
CLASSE = {"rf": RandomForestRegressor, "gb": GradientBoostingRegressor}
ALGORITMO = {"rf": "random_forest", "gb": "gradient_boosting"}

# As sete areas precisam existir no artefato para os dois algoritmos.
areas_artefato = set(arte["areas"])
assert areas_artefato == set(AREAS), (
    f"areas do artefato divergem: faltam {sorted(set(AREAS) - areas_artefato)}, "
    f"sobram {sorted(areas_artefato - set(AREAS))}")
for alg in ("random_forest", "gradient_boosting"):
    faltando = [a for a in AREAS if alg not in arte["areas"][a]]
    assert not faltando, f"{alg} ausente nas areas: {faltando}"
    for a in AREAS:
        assert params(a, alg), f"{a}/{alg}: parametros vazios"
print(f"artefato cobre as {len(AREAS)} areas para random_forest e gradient_boosting")


def iguais(a, b):
    """Comparacao tolerante a float, tratando None explicitamente."""
    if a is None or b is None:
        return a is b
    if isinstance(a, bool) or isinstance(b, bool):
        return a is b
    if isinstance(a, (int, float)) and isinstance(b, (int, float)):
        return abs(float(a) - float(b)) <= 1e-12
    return a == b


def fmt(v):
    """Formata para exibicao; sem isso o pandas coage a coluna a float e None vira NaN."""
    if v is None:
        return "None"
    if isinstance(v, float) and v.is_integer():
        return str(int(v))
    return str(v)


AUSENTE = object()

linhas = []
for area in AREAS:
    for chave, alg in ALGORITMO.items():
        declarados = HIPERPARAMETROS_BASELINE[chave]
        efetivos = CLASSE[chave](**declarados).get_params()   # apenas instancia, nao ajusta
        p_art = params(area, alg)
        for nome in sorted(set(p_art) | set(declarados)):
            v_art = p_art.get(nome, AUSENTE)
            v_base = efetivos.get(nome, AUSENTE)
            linhas.append({
                "area": area,
                "algoritmo": alg,
                "parametro": nome,
                "artefato": "(ausente)" if v_art is AUSENTE else fmt(v_art),
                "baseline": "(ausente)" if v_base is AUSENTE else fmt(v_base),
                "origem_baseline": "declarado" if nome in declarados else "padrao sklearn",
                "difere": "nao" if iguais(v_art, v_base) else "SIM",
            })

comparacao = pd.DataFrame(linhas)
assert comparacao["area"].nunique() == 7, "as 7 areas deveriam estar na comparacao"
assert set(comparacao["algoritmo"]) == {"random_forest", "gradient_boosting"}
print(comparacao.shape[0], "linhas de comparacao (area x algoritmo x parametro)")
pd.set_option("display.max_rows", 90)
comparacao

Como os hiperparâmetros do artefato não variam por área, o padrão de diferenças se repete nas
sete. O resumo abaixo mostra, por algoritmo e parâmetro, os dois valores e se diferem.

In [ ]:
resumo = (comparacao.drop(columns="area")
          .drop_duplicates()
          .sort_values(["algoritmo", "parametro"])
          .reset_index(drop=True))
esperado = comparacao.groupby("algoritmo")["parametro"].nunique().sum()
assert len(resumo) == esperado, "os hiperparametros variam entre areas; o resumo perderia informacao"

n_dif = int((comparacao["difere"] == "SIM").sum())
print(f"{n_dif} de {len(comparacao)} pares (area, algoritmo, parametro) diferem do baseline")
print("parametros que diferem, por algoritmo:",
      resumo[resumo["difere"] == "SIM"].groupby("algoritmo")["parametro"].apply(list).to_dict())
resumo

Leitura do resumo: o Random Forest do artefato é equivalente ao do baseline, porque os três
parâmetros que o artefato explicita são exatamente os padrões que o baseline já usava. O Gradient
Boosting é o que muda de fato, e é onde a comparação com o baseline tem conteúdo.

Duas ressalvas de método a registrar antes do treino. O artefato foi ajustado sobre a formulação em
formato longo por área, com patamar e escala do componente calculados dentro da dobra, enquanto este
notebook treina um modelo por componente em formato wide. E o próprio artefato marca todos os pares
como `configuracao atual`: nenhum sorteio do random search substituiu a configuração de partida.

In [ ]:
origens = pd.DataFrame([
    {"area": a, "algoritmo": alg,
     "origem": arte["areas"][a][alg]["origem"],
     "empate": arte["areas"][a][alg]["empate"],
     "motivo_empate": arte["areas"][a][alg].get("motivo_empate"),
     "MAE_atual": arte["areas"][a][alg]["MAE_atual"],
     "MAE_melhor_sorteio": arte["areas"][a][alg]["MAE_melhor_sorteio"],
     "delta_MAE": arte["areas"][a][alg]["delta_MAE"],
     "piso_area": arte["areas"][a][alg]["piso_area"],
     "passa_significancia": arte["areas"][a][alg]["passa_significancia"]}
    for a in AREAS for alg in ("random_forest", "gradient_boosting")
])
print("origens distintas no artefato:", sorted(set(origens["origem"])))
origens

## 6. Treino por componente com os hiperparâmetros do artefato

Mesma formulação de `treino_rf_gb_por_area.ipynb`: um modelo por componente, 17 dobras
leave-one-group-out, previsões fora-da-dobra empilhadas e o mesmo tratamento de ausentes por
máscara, herdado de `indices_da_dobra`. A única troca é a instanciação, que passa a vir de
`modelo(area, algoritmo)`.

`modelo` é chamado dentro do laço das dobras porque cada dobra precisa de um estimador novo,
exatamente como o notebook de referência constrói um `RandomForestRegressor` por dobra.

In [ ]:
import time
warnings.filterwarnings("ignore")   # o stderr nao deve crescer com um aviso por ajuste


def treinar_area(area, ds, sep):
    """Previsoes fora-da-dobra de RF e GB para cada componente da area, com MAE, RMSE e R2.

    Mesma logica de treinar_area em treino_rf_gb_por_area.ipynb. A unica troca e a
    instanciacao: os estimadores vem de modelo(area, algoritmo), do artefato, em vez
    de RandomForestRegressor(**hiper["rf"]) e GradientBoostingRegressor(**hiper["gb"]).
    """
    df, X, cols_alvo = ds[area], sep[area]["X"], sep[area]["cols_alvo"]
    run = df["run_idx"].to_numpy() if "run_idx" in df.columns else np.arange(len(df))
    previsoes = {}
    for col in cols_alvo:
        y = df[col]
        real, rf, gb, ensaios = [], [], [], []
        for d in DOBRAS:
            idx_tr, idx_val = indices_da_dobra(df, d, y)
            if not idx_tr or not idx_val:
                continue
            X_tr, y_tr, X_val = X.iloc[idx_tr], y.iloc[idx_tr], X.iloc[idx_val]
            m_rf = modelo(area, "random_forest").fit(X_tr, y_tr)
            m_gb = modelo(area, "gradient_boosting").fit(X_tr, y_tr)
            real.extend(y.iloc[idx_val])
            rf.extend(m_rf.predict(X_val))
            gb.extend(m_gb.predict(X_val))
            ensaios.extend(run[idx_val])
        if real:
            real, rf, gb = np.array(real), np.array(rf), np.array(gb)
            # metricas sobre o empilhado das dobras, nao a media por dobra
            previsoes[col] = {"real": real, "rf": rf, "gb": gb, "run_idx": np.array(ensaios)}
    return previsoes


PREVISOES, tempos = {}, {}
t_total = time.perf_counter()
for sufixo in SUFIXOS:
    ds, sep = DADOS[sufixo]
    PREVISOES[sufixo] = {}
    for area in AREAS:
        t0 = time.perf_counter()
        PREVISOES[sufixo][area] = treinar_area(area, ds, sep)
        tempos[(sufixo, area)] = time.perf_counter() - t0
TEMPO_TOTAL = time.perf_counter() - t_total

print(f"tempo total: {TEMPO_TOTAL / 60:.1f} min ({TEMPO_TOTAL:.0f} s)")
pd.DataFrame({
    (sufixo, area): {"componentes": len(PREVISOES[sufixo][area]),
                     "tempo_s": round(tempos[(sufixo, area)], 1)}
    for sufixo in PREVISOES for area in AREAS
}).T

### Gravação das previsões

Mesmo formato e mesma pasta de `previsoes_rf_gb.csv`, para que os notebooks seguintes possam ler os
dois arquivos com o mesmo código. Os asserts conferem as colunas, o total de linhas e, de novo, que
nenhum ensaio de teste chegou às previsões.

In [ ]:
linhas = []
for sufixo, areas in PREVISOES.items():
    for area, comps in areas.items():
        for col, p in comps.items():
            linhas.append(pd.DataFrame({
                "sufixo": sufixo, "area": area, "componente": col,
                "run_idx": p["run_idx"], "real": p["real"], "rf": p["rf"], "gb": p["gb"],
            }))
previsoes = pd.concat(linhas, ignore_index=True)

# grava ao lado dos datasets, onde o resolve() dos notebooks seguintes procura
pasta = os.path.dirname(resolve(PASTA_DADOS, "limites_ativos.csv"))
caminho = os.path.join(pasta, "previsoes_rf_gb_otimizado.csv")
previsoes.to_csv(caminho, index=False)

# mesmo formato do baseline: mesmas colunas, mesma ordem, mesmo numero de linhas
referencia = pd.read_csv(resolve(PASTA_DADOS, "previsoes_rf_gb.csv"))
assert list(previsoes.columns) == list(referencia.columns), "colunas divergem do baseline"
assert len(previsoes) == len(referencia) == 2 * 3277, "numero de previsoes divergente"
assert not set(previsoes["run_idx"]) & RUN_TESTE, "ensaio de teste nas previsoes"

print(caminho)
print(f"{previsoes.shape[0]} linhas, colunas {list(previsoes.columns)}")
previsoes.head(3)

### Métricas

`avaliar` por componente, com o limite de segurança de `limites_ativos.csv` como no baseline, e
`consolidar(grupo="area")` com `sufixo` e `modelo` no laço externo, já que `consolidar` agrupa por
uma coluna só. As médias macro e micro saem juntas.

In [ ]:
def limite_de(col):
    lim = limites["limite_num"].get(col)
    return float(lim) if lim is not None and pd.notna(lim) else None


linhas = []
for (sufixo, area, col), p in previsoes.groupby(["sufixo", "area", "componente"], sort=False):
    for m in ("rf", "gb"):
        linhas.append({"sufixo": sufixo, "area": area, "componente": col, "modelo": m,
                       **avaliar(p["real"], p[m], limite=limite_de(col))})
metricas_componente = pd.DataFrame(linhas)

# consolidar agrupa por uma coluna so; sufixo e modelo entram pelo laco
blocos = []
for (sufixo, m), bloco in metricas_componente.groupby(["sufixo", "modelo"], sort=False):
    blocos.append(consolidar(bloco, grupo="area").assign(sufixo=sufixo, modelo=m))

otimizado = pd.concat(blocos, ignore_index=True)
otimizado = otimizado[["sufixo", "area", "modelo"]
                      + [c for c in otimizado.columns if c not in ("sufixo", "area", "modelo")]]
otimizado = otimizado.sort_values(["sufixo", "area", "modelo"]).reset_index(drop=True)

print(f"{len(metricas_componente)} linhas por componente | {len(otimizado)} linhas por area")
otimizado[["sufixo", "area", "modelo", "componentes", "n_total",
           "MAE_macro", "MAE_micro", "RMSE_macro", "RMSE_micro", "R2_macro", "R2_micro"]].round(3)

### Sanidade: o RF precisa reproduzir o baseline

O Random Forest do artefato é, parâmetro a parâmetro, o mesmo do baseline (seção 5), e o
`random_state` é fixo. Com a mesma partição e a mesma formulação, as previsões têm de ser idênticas,
e o `MAE_macro` tem de coincidir nas sete áreas e nos dois sufixos. Se este assert falhar, a
diferença não está nos hiperparâmetros e sim na formulação ou nos dados, e o resultado do GB não
seria comparável ao baseline.

In [ ]:
baseline = pd.read_csv(resolve(PASTA_DADOS, "baseline_metricas_por_area.csv"))

conf = (baseline[baseline["modelo"] == "rf"][["sufixo", "area", "MAE_macro"]]
        .merge(otimizado[otimizado["modelo"] == "rf"][["sufixo", "area", "MAE_macro"]],
               on=["sufixo", "area"], suffixes=("_baseline", "_otimizado")))
conf["diff_abs"] = (conf["MAE_macro_baseline"] - conf["MAE_macro_otimizado"]).abs()

assert len(conf) == 14, "deveriam ser 7 areas x 2 sufixos"
assert conf["diff_abs"].max() < 1e-6, (
    "o RF do artefato deveria reproduzir o baseline; maior diferenca: "
    f"{conf['diff_abs'].max():.3e}\n"
    + conf.sort_values("diff_abs", ascending=False).head().to_string(index=False))

print(f"assert OK: RF reproduz o baseline nas {len(conf)} combinacoes "
      f"(maior diferenca {conf['diff_abs'].max():.2e}, tolerancia 1e-6)")
conf.round(6)

### Comparação com o baseline

Como o RF é idêntico por construção, toda a diferença está no Gradient Boosting.

In [ ]:
comp = (otimizado[["sufixo", "area", "modelo", "MAE_macro"]]
        .merge(baseline[["sufixo", "area", "modelo", "MAE_macro"]],
               on=["sufixo", "area", "modelo"], suffixes=("_otimizado", "_baseline")))
comp["delta"] = comp["MAE_macro_otimizado"] - comp["MAE_macro_baseline"]   # negativo = melhora

tabela = comp.pivot_table(index="area", columns=["sufixo", "modelo"],
                          values="MAE_macro_otimizado").round(3)
print("MAE_macro do artefato, por area e sufixo:")
print(tabela.to_string())
print()
print("Variacao contra o baseline (negativo = melhora), somente GB:")
print(comp[comp["modelo"] == "gb"]
      .pivot_table(index="area", columns="sufixo", values="delta").round(3).to_string())

Ressalva de método, já registrada na seção 5: os hiperparâmetros foram ajustados sobre a
formulação em formato longo por área, com patamar e escala do componente dentro da dobra, e aqui são
aplicados à formulação por componente em formato wide. Qualquer ganho ou perda do GB nesta tabela é
o efeito de transpor aquela parametrização para esta formulação, não o resultado da busca.

## 7. Comparação com o baseline

Confronto das médias macro de MAE, RMSE e R² entre o baseline e a execução com os hiperparâmetros do
artefato, uma linha por sufixo, área e modelo. As linhas do Random Forest ficam na tabela mesmo com
delta nulo: são elas que mostram que a montagem é a mesma nos dois lados, e sem elas o leitor não
teria como distinguir uma diferença de parametrização de uma diferença de pipeline.

Nada é retreinado aqui. A tabela usa `otimizado`, calculado na seção anterior, e o baseline gravado
em `baseline_metricas_por_area.csv`.

In [ ]:
METRICAS_MACRO = ["MAE_macro", "RMSE_macro", "R2_macro"]
CHAVE = ["sufixo", "area", "modelo"]

comparacao_baseline = (
    baseline[CHAVE + METRICAS_MACRO]
    .merge(otimizado[CHAVE + METRICAS_MACRO], on=CHAVE, suffixes=("_baseline", "_otimizado"))
)
for m in METRICAS_MACRO:
    comparacao_baseline[f"delta_{m}"] = (comparacao_baseline[f"{m}_otimizado"]
                                         - comparacao_baseline[f"{m}_baseline"])

comparacao_baseline = (comparacao_baseline
                       .sort_values(["sufixo", "area", "modelo"])
                       .reset_index(drop=True))

# As 7 areas precisam aparecer nos dois sufixos para os dois modelos.
presenca = comparacao_baseline.groupby(["sufixo", "modelo"])["area"].nunique()
assert len(presenca) == 4, f"deveriam ser 2 sufixos x 2 modelos, ha {len(presenca)}"
assert (presenca == 7).all(), f"area faltando em alguma combinacao:\n{presenca}"
assert len(comparacao_baseline) == 28, f"esperadas 28 linhas, ha {len(comparacao_baseline)}"
assert set(comparacao_baseline["modelo"]) == {"rf", "gb"}

print("assert OK: 7 areas nos dois sufixos para RF e GB |", len(comparacao_baseline), "linhas")
comparacao_baseline.round(3)

### Gravação

Mesma pasta dos demais CSVs de resultado, ao lado de `baseline_metricas_por_area.csv` e
`previsoes_rf_gb_otimizado.csv`.

In [ ]:
pasta = os.path.dirname(resolve(PASTA_DADOS, "limites_ativos.csv"))
caminho_comparacao = os.path.join(pasta, "comparacao_otimizado_baseline.csv")
comparacao_baseline.to_csv(caminho_comparacao, index=False)

print(caminho_comparacao)
print(f"{comparacao_baseline.shape[0]} linhas, {comparacao_baseline.shape[1]} colunas")
print(list(comparacao_baseline.columns))

### Resumo por modelo

In [ ]:
# Resumo do que a tabela mostra, por modelo.
for m in ("rf", "gb"):
    bloco = comparacao_baseline[comparacao_baseline["modelo"] == m]
    d = bloco["delta_MAE_macro"]
    print(f"{m}: piora em {int((d > 1e-6).sum())} de {len(bloco)} | "
          f"melhora em {int((d < -1e-6).sum())} | "
          f"maior |delta| {d.abs().max():.3f}")

print()
print("Delta de MAE_macro do GB, do melhor para o pior:")
print(comparacao_baseline[comparacao_baseline["modelo"] == "gb"]
      [["sufixo", "area", "MAE_macro_baseline", "MAE_macro_otimizado", "delta_MAE_macro"]]
      .sort_values("delta_MAE_macro").round(3).to_string(index=False))

### Leitura dos resultados

O Random Forest reproduz o baseline em todas as catorze combinações de área e sufixo. Os deltas são
nulos a menos de erro de ponto flutuante. Isso era esperado: o artefato entrega, para o Random
Forest, a mesma configuração que o notebook de baseline já usava, e os três parâmetros que ele
explicita coincidem com os padrões do scikit-learn que o baseline herdava. O random_state é o mesmo,
a partição é a mesma e a formulação é a mesma, então as previsões são idênticas.

O Gradient Boosting piorou em treze das catorze combinações. As pioras vão de 0,030 °C, em
CT Knobs e Main Vent Sys com o sufixo `_perfil`, até 0,795 °C em Mainback com `_perfil`, onde o MAE
macro passa de 5,595 para 6,390. Mainback com `_13cat` vem logo atrás, de 5,696 para 6,420. A única
melhora está em IPC Comp com `_13cat`, onde o MAE macro cai de 8,024 para 7,832, uma diferença de
0,192 °C.

A configuração do Gradient Boosting entregue no artefato merece atenção. No artefato, todos os pares de área e
algoritmo estão marcados como configuração atual: nenhuma combinação sorteada pelo random search
substituiu o ponto de partida da busca. A configuração do Gradient Boosting entregue aqui é, portanto,
a configuração de partida, e ela não foi validada por busca alguma na formulação por componente que
este notebook usa. A busca que a examinou rodou sobre a formulação em formato longo por área, com
patamar e escala do componente calculados dentro da dobra, e mesmo lá o resultado foi empate.

Os números acima descrevem o que aconteceu, não explicam por quê. Separar o efeito da parametrização
do efeito da mudança de formulação exigiria uma busca conduzida na formulação por componente, que não
faz parte desta etapa.